# 01 · Train the XGBoost model
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MR-SAIRAM-7/tokenEdge/blob/main/notebooks/01_train_xgboost.ipynb)

This notebook trains the gradient-boosted tree strategy and exports it for the TokenEdge web app.

**What happens**
1. Data is cut off at the end of the **validation** window. The test period is never loaded.
2. Features (technical indicators) at bar *t*; labels resolved after *t* (triple-barrier by default).
3. Chronological train / validation split with **purging** and an **embargo**.
4. Random hyperparameter search scored by **purged K-fold CV** on the training set.
5. Refit with early stopping on validation; trading threshold chosen on validation by **Sharpe after costs**.
6. Export `model.json` + `metadata.json` and download a zip.

CPU is enough for daily data (a few minutes). For hourly crypto, a GPU runtime helps (Runtime → Change runtime type → GPU) with `USE_GPU = True`.

In [ ]:
# --- Setup: works in Google Colab and locally (run from the repo or its notebooks/ folder)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/tokenEdge"):
        !git clone -q https://github.com/MR-SAIRAM-7/tokenEdge.git /content/tokenEdge
    %cd /content/tokenEdge
    !pip install -q -e ".[train]"
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
import tokenedge
print("TokenEdge", tokenedge.__version__)

> If pip upgrades numpy or pandas, Colab may ask to restart. Use **Runtime → Restart session** and run the cells again (the clone and install are skipped the second time).

## Configuration

In [ ]:
SYMBOL = "SPY"                  # e.g. "AAPL", "RELIANCE.NS", "TCS.NS", "BTC/USDT", "ETH/USDT"
SOURCE = "yfinance"             # "yfinance" for stocks/ETFs, "ccxt" for crypto
INTERVAL = "1d"                 # "1d" for stocks, "1h" for crypto
LABEL_METHOD = "triple_barrier" # or "fixed_horizon"
N_TRIALS = 40                   # hyperparameter trials (each = purged 5-fold CV)
USE_GPU = False
REFRESH_DATA = False            # True = download bars newer than the bundled snapshot

## Data

In [ ]:
from tokenedge.config import default_splits, default_label_config, asset_class_of
from tokenedge.data import get_ohlcv, quality_report

df = get_ohlcv(SYMBOL, SOURCE, INTERVAL, refresh=REFRESH_DATA)
splits = default_splits(SOURCE)
print(f"{len(df):,} bars  {df.index[0]} -> {df.index[-1]}")
print("train", splits.train, "| val", splits.val, "| test from", splits.test[0], "| embargo", splits.embargo_bars, "bars")
[i["message"] for i in quality_report(df, INTERVAL, asset_class_of(SOURCE))["issues"]]

## Train

In [ ]:
from tokenedge.ml.xgb_model import train_xgboost

out = train_xgboost(
    df, SYMBOL, SOURCE, INTERVAL,
    label_config=default_label_config(LABEL_METHOD, INTERVAL),
    n_trials=N_TRIALS,
    device="cuda" if USE_GPU else "cpu",
)
md = out.metadata
print("model_id:", md["model_id"])

## Validation results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

val = md["metrics"]["val"]
display(pd.Series({k: val[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "log_loss", "base_rate", "sharpe_after_costs"]}).round(4))
print("splits used:", md["splits"]["train_used"], md["splits"]["val_used"])
print("chosen threshold:", md["threshold"], "| trials counted for DSR:", md["cv"]["trials"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t = pd.DataFrame(md["threshold_search"])
axes[0].plot(t["threshold"], t["sharpe"], marker="o", color="#eb6834")
axes[0].axvline(md["threshold"]["long"], color="gray", lw=1)
axes[0].set_title("Validation Sharpe after costs vs threshold"); axes[0].set_xlabel("P(up) threshold")
imp = pd.Series(md["feature_importance"]).head(15)[::-1]
axes[1].barh(imp.index, imp.values, color="#eb6834")
axes[1].set_title("Feature importance (gain share)")
plt.tight_layout(); plt.show()

## Optional: one final look at the test window

Everything above used only train and validation data. Run this **once**, after you have stopped changing settings: every extra look at the test window turns it into a validation set. The web app's **Compare** page runs the same evaluation with more statistics.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    from tokenedge.ml.registry import ModelRegistry, set_registry
    from tokenedge.experiments import run_market
    set_registry(ModelRegistry(out.model_dir.parent))
    run = run_market(SYMBOL, SOURCE, INTERVAL)
    print(run.report["verdict"])
    display(pd.DataFrame(run.report["metrics"]).T[["total_return", "sharpe", "max_drawdown", "psr", "trades"]].round(3))

## Export

In [ ]:
from tokenedge.ml.training import zip_model_dir

archive = zip_model_dir(out.model_dir)
print("created", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))

## Next steps

1. Unzip the downloaded file into the project's `models/` folder, so you have `models/<model_id>/model.json` and `metadata.json`.
2. Restart or refresh the web app. The model appears on the **Models** page and can be used in **Backtest Lab** and **Compare**.
3. Commit the folder so the deployed app has it: `git add models/<model_id> && git commit -m "Add model" && git push`.